# Figure generation — Edge-IIoTset leakage study
Reproduces Figures 1-6 of the manuscript *"Correlation-Based Leakage Screening Is Insufficient for IoT/IIoT Intrusion-Detection Benchmarks: Redundant Shortcut Encoding in Edge-IIoTset."*

**Requirements:** `ML-EdgeIIoT-dataset.csv` in the working directory (upload it via the Files panel).
All figures use the same Python/scikit-learn pipeline as the numerical results, ensuring figure-table consistency. Random seed = 42 throughout.

| Cell | Produces |
|------|----------|
| 1 | Setup + data loading + preprocessing (152,245 records) |
| 2 | Trains binary + multi-class Random Forests |
| 3 | **Figure 1** - ROC curves (leakage-controlled) |
| 4 | **Figure 2** - Binary confusion matrix (one false positive) |
| 5 | **Figure 3** - Feature-ablation cascade |
| 6 | **Figure 4** - Multi-class confusion matrix (15 classes) |
| 7 | **Figure 5** - Top-15 Gini feature importance |
| 8 | **Figure 6** - SHAP summary (n = 1000) |


In [ ]:
# ===== CELL 1 - Setup, load, preprocess =====
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (confusion_matrix, roc_curve, roc_auc_score,
                             accuracy_score, f1_score)
RS = 42
np.random.seed(RS)
plt.rcParams.update({'font.size': 13, 'axes.titlesize': 14, 'axes.labelsize': 13,
                     'xtick.labelsize': 11, 'ytick.labelsize': 11})

CSV = 'ML-EdgeIIoT-dataset.csv'   # ensure this file is in the Files panel
df = pd.read_csv(CSV, low_memory=False)
print('Loaded:', df.shape, '(expected 157800 x 63)')

DROP13 = ['frame.time','ip.src_host','ip.dst_host','arp.dst.proto_ipv4','arp.src.proto_ipv4',
          'http.file_data','http.request.uri.query','http.request.full_uri','http.referer',
          'tcp.options','tcp.payload','tcp.srcport','mqtt.msg']
df = df.drop(columns=[c for c in DROP13 if c in df.columns]).dropna().drop_duplicates().reset_index(drop=True)
assert len(df) == 152245, f'expected 152,245, got {len(df):,}'
print('After clean:', len(df), '-> CHECKPOINT PASSED')

CAT = ['http.request.method','http.request.version','dns.qry.name.len',
       'mqtt.conack.flags','mqtt.protoname','mqtt.topic']
for c in CAT:
    df[c] = df[c].astype('category').cat.codes

LEAK = ['dns.qry.name.len','mqtt.topic','mqtt.protoname','mqtt.conack.flags']
X = df.drop(columns=['Attack_label','Attack_type'])
cols = [c for c in X.columns if c not in LEAK]     # 44 leakage-controlled features
yb   = df['Attack_label'].astype(int)              # binary target
ymul = df['Attack_type']                           # 15-class target
print('Controlled features:', len(cols))

In [ ]:
# ===== CELL 2 - Train models (binary + multi-class) =====
Xtrb, Xteb, ytrb, yteb = train_test_split(X[cols], yb, test_size=.2, random_state=RS, stratify=ymul)
rfb = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=RS, n_jobs=-1).fit(Xtrb, ytrb)

sc = StandardScaler().fit(Xtrb)
logreg = LogisticRegression(penalty='l2', max_iter=2000, random_state=RS).fit(sc.transform(Xtrb), ytrb)
mlp    = MLPClassifier(hidden_layer_sizes=(64,32), early_stopping=True, random_state=RS).fit(sc.transform(Xtrb), ytrb)

Xtrm, Xtem, ytrm, ytem = train_test_split(X[cols], ymul, test_size=.2, random_state=RS, stratify=ymul)
rfm = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=RS, n_jobs=-1).fit(Xtrm, ytrm)

print('Binary RF acc:', round(accuracy_score(yteb, rfb.predict(Xteb)), 4))
print('Multiclass macro-F1:', round(f1_score(ytem, rfm.predict(Xtem), average='macro'), 3), '(expected ~0.947)')

In [ ]:
# ===== CELL 3 - FIGURE 1: ROC curves =====
plt.figure(figsize=(7.5, 6))
for name, prob in [('Random Forest', rfb.predict_proba(Xteb)[:,1]),
                   ('Logistic Regression', logreg.predict_proba(sc.transform(Xteb))[:,1]),
                   ('MLP', mlp.predict_proba(sc.transform(Xteb))[:,1])]:
    fpr, tpr, _ = roc_curve(yteb, prob)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC={roc_auc_score(yteb, prob):.4f})')
plt.plot([0,1], [0,1], 'k--', lw=1, alpha=.5)
plt.xlabel('False positive rate'); plt.ylabel('True positive rate')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig('Figure1_ROC.png', dpi=300); plt.show()

In [ ]:
# ===== CELL 4 - FIGURE 2: Binary confusion matrix =====
cm = confusion_matrix(yteb, rfb.predict(Xteb))
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Normal','Attack'], yticklabels=['Normal','Attack'],
            annot_kws={'size': 16})
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.tight_layout(); plt.savefig('Figure2_binary_confusion.png', dpi=300); plt.show()
tn, fp, fn, tp = cm.ravel()
print(f'TN={tn} FP={fp} FN={fn} TP={tp}  (errors: {fp+fn})')

In [ ]:
# ===== CELL 5 - FIGURE 3: Feature-ablation cascade =====
order = pd.Series(rfb.feature_importances_, index=cols).sort_values(ascending=False).index.tolist()
ks = [0,2,4,8,12,16,20,24,28,32,36,40]
acc = []
for k in ks:
    keep = order[k:]
    if len(keep) < 2: break
    m = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=RS, n_jobs=-1).fit(Xtrb[keep], ytrb)
    acc.append(accuracy_score(yteb, m.predict(Xteb[keep])))
plt.figure(figsize=(7.2, 4.8))
plt.plot(ks[:len(acc)], acc, 'o-', color='#c0392b', lw=2, markersize=6)
plt.axhline(0.84, ls='--', color='gray', lw=1, alpha=.7)
plt.xlabel('Number of top Random Forest features removed')
plt.ylabel('Binary accuracy (leakage-controlled)')
plt.ylim(0.80, 1.01); plt.grid(alpha=.3); plt.tight_layout()
plt.savefig('Figure3_ablation.png', dpi=300); plt.show()

In [ ]:
# ===== CELL 6 - FIGURE 4: Multi-class confusion matrix =====
labels = sorted(ymul.unique())
cmm = confusion_matrix(ytem, rfm.predict(Xtem), labels=labels)
plt.figure(figsize=(10, 8))
sns.heatmap(cmm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=labels, yticklabels=labels, annot_kws={'size': 7})
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right', fontsize=8); plt.yticks(fontsize=8)
plt.tight_layout(); plt.savefig('Figure4_multiclass_confusion.png', dpi=300); plt.show()

In [ ]:
# ===== CELL 7 - FIGURE 5: Top-15 Gini feature importance =====
imp = pd.Series(rfb.feature_importances_, index=cols).sort_values(ascending=False)
print('Top 5 (should match Table 9):'); print(imp.head(5).round(3).to_string())
plt.figure(figsize=(8, 6))
imp.head(15).iloc[::-1].plot(kind='barh', color='#2c5f8a')
plt.xlabel('Gini importance'); plt.ylabel(None)
plt.tight_layout(); plt.savefig('Figure5_feature_importance.png', dpi=300); plt.show()

In [ ]:
# ===== CELL 8 - FIGURE 6: SHAP summary (n = 1000) =====
# ~5-8 minutes. Reuses the binary RF (rfb) and test set (Xteb).
import shap
samp = Xteb.sample(n=1000, random_state=RS)
sv = shap.TreeExplainer(rfb).shap_values(samp)
sv_pos = sv[:,:,1] if isinstance(sv, np.ndarray) and sv.ndim == 3 else (sv[1] if isinstance(sv, list) else sv)
shap.summary_plot(sv_pos, samp, show=False, max_display=15)
plt.tight_layout(); plt.savefig('Figure6_shap_summary.png', dpi=300, bbox_inches='tight'); plt.show()
print('Figure 6 (SHAP, n=1000) saved.')